# QAOA on Max-Cut (primer)

*Part of milestone 3 · borrowed from UTC's problem set (Problem 2)*

Max-Cut is the textbook QAOA problem: split a small grid's substations into two groups so that as many lines as possible run between the groups. (It's only a practice problem. Real grid islanding is closer to the opposite: cut as few lines as possible.) It shows QAOA's three pieces, which you'll meet again in plant scheduling: a cost layer, a mixer, and a classical optimizer that tunes them.

**Your job (about 45 minutes):** fill in the cost layer and the optimizer loop, solve the 6-node grid, and compare it with the brute-force best. The 12-node scale-up is optional.

> **Before you run anything, pick the kernel.** When this notebook opens, a **Select Kernel** box appears with *Python 3 (ipykernel)* already selected. That's the wrong one: change it to **qBraid-SDK (v0.12.2)**, then click **Select**. Picked the wrong one, or closed the box? Click the kernel name at the top right of the notebook and choose **qBraid-SDK (v0.12.2)**. The first cell checks, and tells you if it's wrong.

In [1]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")

# Qiskit Aer is Qiskit's simulator. It runs right here in your Lab instance: instant, free and noise-free,
# with no job and no credits. The first run installs it (a few seconds), pinned to the version tested here.
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

sim = AerSimulator()

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.


In [25]:
# The grids (nodes are substations, edges are power lines)

# 6-node grid (2 x 3) for Part 1
edges_small = [(0,1),(1,2),(3,4),(4,5),(0,3),(1,4),(2,5)]
n_small = 6

# 12-node grid (3 x 4) for the GPU scale-up in Part 2
edges_large = [(0,1),(1,2),(2,3),(4,5),(5,6),(6,7),(8,9),(9,10),(10,11),
               (0,4),(4,8),(1,5),(5,9),(2,6),(6,10),(3,7),(7,11)]
n_large = 12

n, edges = n_small, edges_small    # Part 1 uses the 6-node grid; the scale-up cell switches to 12


In [26]:
def cut_value(bitstring, edges):
    """Count how many edges are cut by a 0/1 assignment (given as a string)."""
    bits = [int(b) for b in bitstring]
    return sum(1 for (u, v) in edges if bits[u] != bits[v])


In [27]:
def qaoa_circuit(params, n, edges, p):
    """Build a p-layer QAOA circuit. params = [gammas..., betas...]."""
    gammas = params[:p]
    betas = params[p:]
    qc = QuantumCircuit(n, n)
    qc.h(range(n))                    # start in equal superposition
    for layer in range(p):
        # --- Cost layer ---
        for (u, v) in edges:
            qc.cx(u, v)                       # v now holds (u XOR v): 1 if the edge is cut
            qc.rz(2 * gammas[layer], v)       # phase depends on cut / not cut
            qc.cx(u, v)                       # undo, restoring v

        # --- Mixer layer (provided) ---
        for q in range(n):
            qc.rx(2 * betas[layer], q)
    qc.measure(range(n), range(n))
    return qc

In [28]:
def expected_cut(params, n, edges, p, shots=512):
    """Run the circuit and return the AVERAGE cut value (we maximize this)."""
    qc = qaoa_circuit(params, n, edges, p)
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    total = sum(counts.values())
    # bitstrings come back reversed in Qiskit; reverse to match node order
    avg = sum(cut_value(bs[::-1], edges) * c for bs, c in counts.items()) / total
    return avg


In [19]:
# Optimize (we minimize the NEGATIVE average cut)
p = 1                                  # number of QAOA layers; try 1, then 2
# Uses n, edges as last set: the 6-node grid at first, the 12-node grid after the scale-up cell

def objective(params):
    return -expected_cut(params, n, edges, p)

# Several random starts, keep the best. One start can stall in a poor local optimum.
rng = np.random.default_rng(0)
N_STARTS = 5
result = None
for s in range(N_STARTS):
    # gammas in [0, pi], betas in [0, pi/2]: the ranges where the angles actually matter
    x0 = np.concatenate([rng.uniform(0, np.pi, p), rng.uniform(0, np.pi / 2, p)])
    res = minimize(objective, x0, method="COBYLA", options={"maxiter": 100})
    print(f"start {s}: average cut {-res.fun:.3f}   angles {np.round(res.x, 3)}")
    if result is None or res.fun < result.fun:
        result = res

print(f"\nBest: average cut {-result.fun:.3f} of {len(edges)} lines, angles {np.round(result.x, 3)}")

start 0: average cut 5.164   angles [ 3.488 -0.377]
start 1: average cut 3.990   angles [1.144 0.228]
start 2: average cut 4.969   angles [3.516 1.29 ]
start 3: average cut 3.924   angles [1.856 1.152]
start 4: average cut 5.035   angles [0.28 2.8 ]

Best: average cut 5.164 of 7 lines, angles [ 3.488 -0.377]


In [21]:
# Read out the best partition (run after the optimizer works)
best_params = result.x
qc = qaoa_circuit(best_params, n, edges, p)
counts = sim.run(transpile(qc, sim), shots=2048).result().get_counts()
# Of the bitstrings QAOA sampled, keep the one with the biggest cut.
# Qiskit lists qubit 0 last, so reverse each bitstring to match node order.
best_bits = max((bs[::-1] for bs in counts), key=lambda b: cut_value(b, edges))
group_A = [i for i, b in enumerate(best_bits) if b == '0']
group_B = [i for i, b in enumerate(best_bits) if b == '1']
print("Group A:", group_A)
print("Group B:", group_B)
print("Cut value:", cut_value(best_bits, edges), "(compare with the brute-force best below)")
# On a grid this small, the best of 2,048 samples finds the best cut even by random guessing.
# So also compare the AVERAGE cut per shot with random guessing, which cuts each line half the time.
avg = sum(cut_value(bs[::-1], edges) * c for bs, c in counts.items()) / sum(counts.values())
print(f"Average cut per shot: {avg:.2f}   random guessing: {len(edges) / 2:.1f}")

Group A: [0, 2, 4]
Group B: [1, 3, 5]
Cut value: 7 (compare with the brute-force best below)
Average cut per shot: 5.01   random guessing: 3.5


In [22]:
# Classical baseline (brute force), provided. Fine up to about 20 nodes.
from itertools import product
def brute_force(n, edges):
    best_cut, best = -1, None
    for assign in product([0,1], repeat=n):
        c = cut_value("".join(map(str, assign)), edges)
        if c > best_cut:
            best_cut, best = c, assign
    return best_cut, best
print(f"Best possible cut (classical, {n} nodes):", brute_force(n, edges)[0])


Best possible cut (classical, 6 nodes): 7


### Optional: 12-node scale-up (Part 2)

UTC asks you to run a bigger grid on qBraid's GPU simulator and note how the runtime changes. Two steps:

1. Run the next cell to switch to the 12-node grid, then re-run the optimizer and read-out cells above. Keep tuning on Aer: the optimizer runs the circuit dozens of times, and 12 qubits still takes seconds here.
2. Then run the cell after it. It sends your tuned circuit to qBraid's GPU simulator (an NVIDIA GH200) and times it against Aer.

In [23]:
# Step 1: switch to the 12-node grid, then re-run the optimizer and read-out cells above
n, edges = n_large, edges_large
print(f"Now on the {n}-node grid ({len(edges)} lines). Re-run the optimizer and read-out cells, then run the next cell.")

Now on the 12-node grid (17 lines). Re-run the optimizer and read-out cells, then run the next cell.


In [24]:
# Step 2 (optional): run your tuned circuit on qBraid's GPU simulator and time it against Aer.
# Free up to a daily limit of runs, then 5 credits a minute of run time. Set this to True to send it:
RUN_ON_GPU = False

import time
qc = qaoa_circuit(result.x, n, edges, p)
t0 = time.time()
aer_counts = sim.run(transpile(qc, sim), shots=2048).result().get_counts()
aer_secs = time.time() - t0

if RUN_ON_GPU:
    from qbraid.runtime import QbraidProvider
    device = QbraidProvider().get_device("qbraid:qbraid:sim:gpu-sv")
    t0 = time.time()
    job = device.run(qc, shots=2048)
    gpu_counts = job.result().data.get_counts()
    gpu_secs = time.time() - t0
    print("Job ID:", job.id)          # copy this into your write-up
    for name, c in [("Aer, here", aer_counts), ("qBraid GPU simulator", gpu_counts)]:
        avg = sum(cut_value(bs[::-1], edges) * k for bs, k in c.items()) / sum(c.values())
        print(f"{name}: average cut per shot {avg:.2f}")
    print(f"{n} qubits, 2,048 shots: Aer took {aer_secs:.2f} s here; the GPU run took {gpu_secs:.0f} s, round trip included")
else:
    print(f"Aer ran the {n}-qubit circuit (2,048 shots) in {aer_secs:.2f} s. Set RUN_ON_GPU = True to compare with qBraid's GPU simulator.")

# When does a GPU pay off? Each extra qubit doubles the memory a state-vector simulator needs:
# Aer on a 4 GB Lab instance runs out at about 28 qubits, and qBraid's GPU simulator goes to 35.

Aer ran the 12-qubit circuit (2,048 shots) in 0.10 s. Set RUN_ON_GPU = True to compare with qBraid's GPU simulator.


## What to have ready (milestone 3)
- The 6-node partition and its cut value, next to the brute-force best.
- Your average cut per shot next to random guessing (half the lines), and why the average is the fairer measure here. (Can you find a best partition just by looking at the grid? How do you know it's best?)
- One sentence for your deck: what the cost layer, the mixer and the optimizer each do, and how the Max-Cut cost relates to your track's cost function.
- Optional: the 12-node result (Part 2), with Aer's time next to the GPU simulator's, and what that says about when a GPU pays off.